In [1]:
import os
print(os.getcwd())

/home/aychen/rubricsgen/LLMEval-Med


In [5]:
import json

input_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset.json"
output_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_filtered.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

def has_nonempty_checklist(item):
    if not isinstance(item, dict):
        return False
    if "checklist" not in item:
        return False
    val = item["checklist"]
    # 非空字符串/非空对象均视为有效；过滤 None、空串、只含空白、"NaN"
    if val is None:
        return False
    if isinstance(val, str):
        return val.strip() != "" and val.strip().lower() != "nan"
    # 如果是列表/字典等，也要求非空
    if isinstance(val, (list, dict)):
        return len(val) > 0
    # 其他类型（数字/布尔等）按存在视为有效
    return True

# 递归收集所有满足条件的字典条目
collected = []
def collect(obj):
    if isinstance(obj, list):
        for x in obj:
            collect(x)
    elif isinstance(obj, dict):
        if has_nonempty_checklist(obj):
            collected.append(obj)
        # 继续深入子字段
        for v in obj.values():
            collect(v)

collect(data)

with open(output_file, "w", encoding="utf-8") as f:
    for item in collected:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ 已写出 {len(collected)} 条包含非空 checklist 的记录到 {output_file}")


✅ 已写出 181 条包含非空 checklist 的记录到 /home/aychen/rubricsgen/LLMEval-Med/data/dataset_filtered.jsonl


In [11]:
import json
from collections import defaultdict

input_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_filtered.jsonl"

category_map = defaultdict(set)
total, bad = 0, 0

def collect(obj):
    # 递归收集 {category1 -> category2}
    if isinstance(obj, list):
        for x in obj:
            collect(x)
    elif isinstance(obj, dict):
        c1 = obj.get("category1")
        c2 = obj.get("category2")
        if c1 and c2:
            category_map[c1].add(c2)
        for v in obj.values():
            collect(v)

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total += 1
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            bad += 1
            continue
        collect(obj)

print(f"✅ 解析成功 {total - bad}/{total} 行")
for c1, c2s in category_map.items():
    print(f"\n📂 category1 = {c1}（{len(c2s)} 种 category2）")
    for c2 in sorted(c2s):
        print(f"   - {c2}")


✅ 解析成功 139/139 行

📂 category1 = 医疗知识（8 种 category2）
   - 健康养生
   - 其他医疗信息
   - 医学基础知识/医学考试
   - 手术/操作/治疗方式
   - 检验/检查
   - 疾病相关
   - 症状相关
   - 药物相关

📂 category1 = 一般语言理解（1 种 category2）
   - 多轮对话


In [15]:
import json
import re
from collections import Counter

input_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_filtered.jsonl"

# 统计单条 checklist 里的“点数”
def count_checklist_points(text: str) -> int:
    if not isinstance(text, str):
        return 0
    # 常见编号样式：1. / 1、 / 1) / （1） / (1)
    pattern = r"(?m)^\s*(?:\(|（)?\d+(?:\)|）|\.|、)"
    return len(re.findall(pattern, text))

problem_points = []         # [(problem, n_points), ...]
total, bad = 0, 0

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        total += 1
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            bad += 1
            continue

        # 只统计同时包含 problem 和 checklist 的条目
        checklist = obj.get("checklist")
        problem = obj.get("problem")
        if problem and checklist:
            n = count_checklist_points(checklist)
            problem_points.append((problem, n))

# 分布统计
distribution = Counter(n for _, n in problem_points)

print(f"✅ 解析成功 {total - bad}/{total} 行；统计到 {len(problem_points)} 条有 checklist 的记录")
print("\n📊 checklist 点数分布：")
for n in sorted(distribution):
    print(f"{n} 点: {distribution[n]} 个 problem")

✅ 解析成功 130/130 行；统计到 130 条有 checklist 的记录

📊 checklist 点数分布：
2 点: 2 个 problem
3 点: 54 个 problem
4 点: 61 个 problem
5 点: 8 个 problem
6 点: 3 个 problem
7 点: 2 个 problem


In [16]:
import json
import re

input_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_filtered.jsonl"
output_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_checklist_3-4.jsonl"

def count_checklist_points(text: str) -> int:
    if not isinstance(text, str):
        return 0
    # 匹配常见编号：1. / 1、 / 1) / （1） / (1)
    pattern = r"(?m)^\s*(?:\(|（)?\d+(?:\)|）|\.|、)"
    return len(re.findall(pattern, text))

kept = []
total = 0

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        total += 1
        checklist = obj.get("checklist")
        problem = obj.get("problem")
        if problem and checklist:
            n = count_checklist_points(checklist)
            if 3 <= n <= 4:
                obj["_checklist_points"] = n  # 可选：保留点数信息
                kept.append(obj)

with open(output_file, "w", encoding="utf-8") as f:
    for obj in kept:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"✅ 总共 {total} 条，筛选出 {len(kept)} 条含 3–4 点 checklist 的 problem，已保存到 {output_file}")


✅ 总共 130 条，筛选出 115 条含 3–4 点 checklist 的 problem，已保存到 /home/aychen/rubricsgen/LLMEval-Med/data/dataset_checklist_3-4.jsonl


In [18]:
import json
import re

input_file = "/home/aychen/rubricsgen/LLMEval-Med/data/dataset_checklist_3-4.jsonl"
output_file = "/home/aychen/rubricsgen/LLMEval-Med/data/problems_checklist_clean.jsonl"

def split_checklist(text: str):
    """把 checklist 拆成按点的列表，并清理提示性字样"""
    if not isinstance(text, str):
        return []
    # 分割编号：1. / 1、 / 1) / （1）
    items = re.split(r"(?m)^\s*(?:\(|（)?\d+(?:\)|）|\.|、)", text)
    clean_items = []
    for item in items:
        s = item.strip()
        if not s:
            continue
        # 去掉提示性前缀
        s = re.sub(r"(内容要求：|次要需求：?|核心需求：?)", "", s)
        # 再去掉多余空格
        s = s.strip()
        if s:
            clean_items.append(s)
    return clean_items

kept = []

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        problem = obj.get("problem")
        difficulty = obj.get("difficulty")
        checklist = obj.get("checklist")

        if problem and difficulty and checklist:
            new_obj = {
                "problem": problem,
                "difficulty": difficulty,
                "checklist": split_checklist(checklist)
            }
            kept.append(new_obj)

with open(output_file, "w", encoding="utf-8") as f:
    for obj in kept:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"✅ 已提取并清理 {len(kept)} 条，结果保存到 {output_file}")

✅ 已提取并清理 115 条，结果保存到 /home/aychen/rubricsgen/LLMEval-Med/data/problems_checklist_clean.jsonl
